<a href="https://colab.research.google.com/github/Xpercussionheadx/Braden/blob/main/Minecraft_Skin_Generator_NO_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Skin Generator NO UI

Credits:
- Author of the AI Model: Cory Spencer <cory@monadical.com>
- Forked Improved Version & Ports: [Nick088](https://linktr.ee/Nick088)
- daroche (helping me fix the 3d model texture isue)
- [Brottweiler](https://gist.github.com/Brottweiler/483d0856c6692ef70cf90bf1a85ce364) (script to fix the 3d model texture
- [meew](https://huggingface.co/spaces/meeww/Minecraft_Skin_Generator/blob/main/models/player_model.glb) (Minecraft Player 3d model)

In [1]:
#@title Install
#@markdown If you get a restart runtime warning, you don't have to, just click cancel
!git clone https://github.com/Nick088Official/Minecraft_Skin_Generator.git
%cd Minecraft_Skin_Generator
!pip install -r /content/Minecraft_Skin_Generator/Scripts/requirements_no_ui.txt
!apt-get install imagemagick
from IPython.display import clear_output
from google.colab import files
clear_output()
print("Installed!")

Installed!


In [2]:
#@title Run Inference

#@markdown Your Prompt
prompt = "An agent Spy" #@param {type:"string"}

#@markdown Which Stable Diffusion Model to use for inferencing, xl understands prompts better
stable_diffusion_model = 'xl' #@param ['2', 'xl']

#@markdown The number of denoising steps of the image. More denoising steps usually lead to a higher quality image at the cost of slower inference
num_inference_steps = 25 #@param {type:"integer"}

#@markdown How closely the generated image adheres to the prompt
guidance_scale = 7.5 #@param {type:"number"}

#@markdown The precision type to load the model, like fp16 which is faster, or fp32 which is more precise but more resource consuming and requires paid gpu
model_precision_type = 'fp16' #@param ['fp16', 'fp32']

#@markdown A starting point to initiate generationg, put 0 for random
seed = 42 #@param {type:"integer"}

#@markdown The name of the output skin image asset
filename = "output-skin.png" #@param {type:"string"}

#@markdown View the 3D Model of the skin too
model_3d = True #@param {type:"boolean"}

#@markdown Produce verbose (detailed) output while running
verbose = False #@param {type:"boolean"}

if verbose:
  verbose_opt = '--verbose'
else:
  verbose_opt = ''

if model_3d:
  model_3d_opt = '--model_3d'
else:
  model_3d_opt = ''

if stable_diffusion_model == '2':
  sd_model = "minecraft-skins"
else:
  sd_model = "minecraft-skins-sdxl"

command = f"Scripts/{sd_model}.py '{prompt}' {num_inference_steps} {guidance_scale} {model_precision_type} {seed} {filename} {model_3d_opt} {verbose_opt}"

!python $command

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
No CUDA or MPS devices found, running on CPU.
model_index.json: 100% 645/645 [00:00<00:00, 2.63MB/s]
Fetching 20 files: 100% 20/20 [02:33<00:00,  7.69s/it]
Download complete: : 12.1GB [02:33, 106MB/s]              
Loading pipeline components...:   0% 0/7 [00:00<?, ?it/s]
Download complete: : 12.1GB [02:34, 78.1MB/s]

Loading pipeline components...:  43% 3/7 [00:02<00:03,  1.03it/s]
Loading weights: 100% 196/196 [00:00<00:00, 653.83it/s, Materializing param=text_model.final_layer_norm.weight]

Loading weights: 100% 517/517 [00:00<00:00, 662.12it/s, Materializing param=text_projection.weight]

Loading pipeline components...:  86% 6/7 [00:04<00:00,  1.44it/s]
Loading pipeline components...: 100%

In [3]:
#@title Display the Generated Minecraft Skin Asset

import matplotlib.pyplot as plt

img = plt.imread(f"output_minecraft_skins/{filename}")

plt.axis("off")

plt.imshow(img)

FileNotFoundError: [Errno 2] No such file or directory: 'output_minecraft_skins/output-skin.png'

In [ ]:
#@title Download the Generated Minecraft Skin Asset

files.download(f"output_minecraft_skins/{filename}")

In [ ]:
#@title Download the Generated Minecraft Skin 3D Model (only if you checked the model_3d box)

files.download(f"output_minecraft_skins/{filename}")

In [ ]:
# 1. Install required packages
!pip install flask flask-cors pyngrok

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import base64
import subprocess
import os

app = Flask(__name__)
CORS(app) # Allow the frontend to talk to this backend

# 2. Set up Ngrok
# Note: Be careful sharing this token publicly!
ngrok.set_auth_token("3C0q74X0ocPExrU2S9UXzL7F1eg_3g1ZGsBKgwLqbX3mhC1Fs")
public_url = ngrok.connect(5000).public_url
print(f"✅ COPY THIS URL TO YOUR APP: {public_url}")

@app.route('/generate_skin', methods=['POST'])
def generate():
    data = request.json
    prompt = data.get('prompt', 'A default skin')

    print(f"Generating skin for: {prompt}")

    # Configuration (matching the notebook defaults)
    stable_diffusion_model = 'xl'
    num_inference_steps = 25
    guidance_scale = 7.5
    model_precision_type = 'fp16'
    seed = 0 # 0 for random seed every time
    filename = "api-output.png"

    sd_model = "minecraft-skins-sdxl" if stable_diffusion_model == 'xl' else "minecraft-skins"

    # 3. Construct the command exactly as the notebook does
    command = f"python Scripts/{sd_model}.py '{prompt}' {num_inference_steps} {guidance_scale} {model_precision_type} {seed} {filename}"

    try:
        # 4. Run the command line script
        print(f"Running command: {command}")
        subprocess.run(command, shell=True, check=True)

        # 5. Read the generated image from the output folder
        output_path = f"output_minecraft_skins/{filename}"

        if not os.path.exists(output_path):
            return jsonify({"error": "Image generation failed, file not found."}), 500

        with open(output_path, "rb") as image_file:
            img_str = base64.b64encode(image_file.read()).decode("utf-8")

        return jsonify({"image": img_str})

    except Exception as e:
        print(f"Error generating skin: {e}")
        return jsonify({"error": str(e)}), 500

# Start the server!
app.run(port=5000)